In [1]:
# Spark Operations 

In [2]:
from pyspark.sql import SparkSession

In [3]:
spark =SparkSession.builder.appName("RDD_Operatoin").getOrCreate()

26/08/21 09:30:19 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [11]:
costemer_data = [
    "customer_id,name,city,state,country,registration_date,is_active",
    "0,Customer_0,Bangalore,Karnataka,India,2023-11-11,True",
    "1,Customer_1,Hyderabad,Delhi,India,2023-08-26,True",
    "2,Customer_2,Ahmedabad,West Bengal,India,2023-06-23,True",
    "3,Customer_3,Bangalore,Tamil Nadu,India,2023-03-24,False",
    "4,Customer_4,Bangalore,Gujarat,India,2023-06-06,False",
    "5,Customer_5,Delhi,Maharashtra,India,2023-04-19,False"
]

In [12]:
data_rdd = spark.sparkContext.parallelize(costemer_data)

In [13]:
data_rdd.getNumPartitions()

2

In [33]:
# RDD - Rsilient Distributed Dataset

In [34]:
# first() : returns the first element of RDD

In [35]:
header = data_rdd.first()

In [36]:
header

'customer_id,name,city,state,country,registration_date,is_active'

# Filter() -> based on the condition it's going to transform the dataset

In [37]:
data_rdd = data_rdd.filter(lambda row : row!=header) # this is a Transformation

In [38]:
data_rdd.collect()

['0,Customer_0,Bangalore,Karnataka,India,2023-11-11,True',
 '1,Customer_1,Hyderabad,Delhi,India,2023-08-26,True',
 '2,Customer_2,Ahmedabad,West Bengal,India,2023-06-23,True',
 '3,Customer_3,Bangalore,Tamil Nadu,India,2023-03-24,False',
 '4,Customer_4,Bangalore,Gujarat,India,2023-06-06,False',
 '5,Customer_5,Delhi,Maharashtra,India,2023-04-19,False']

In [39]:
def parse_row(row):
    fields = row.split(',')
    return (
    int(fields[0]),
        fields[1],
        fields[2],
        fields[3],
        fields[4],
        fields[5],
        fields[6]=='True'
    )

In [40]:
parsed_rdd = data_rdd.map(parse_row)

In [42]:
parsed_rdd.collect()

[(0, 'Customer_0', 'Bangalore', 'Karnataka', 'India', '2023-11-11', True),
 (1, 'Customer_1', 'Hyderabad', 'Delhi', 'India', '2023-08-26', True),
 (2, 'Customer_2', 'Ahmedabad', 'West Bengal', 'India', '2023-06-23', True),
 (3, 'Customer_3', 'Bangalore', 'Tamil Nadu', 'India', '2023-03-24', False),
 (4, 'Customer_4', 'Bangalore', 'Gujarat', 'India', '2023-06-06', False),
 (5, 'Customer_5', 'Delhi', 'Maharashtra', 'India', '2023-04-19', False)]

# Advance RDD Operations

In [43]:
# Extract a field with Map() - City and the state

In [44]:
name_city_rdd = parsed_rdd.map(lambda row:(row[2],row[3]))

In [45]:
name_city_rdd.collect()

[('Bangalore', 'Karnataka'),
 ('Hyderabad', 'Delhi'),
 ('Ahmedabad', 'West Bengal'),
 ('Bangalore', 'Tamil Nadu'),
 ('Bangalore', 'Gujarat'),
 ('Delhi', 'Maharashtra')]

In [46]:
# we went to filter the active costumers 

In [48]:
active_costumers_rdd = parsed_rdd.filter(lambda row :row[6]==True)

In [63]:
active_costumers_rdd.collect()

[(0, 'Customer_0', 'Bangalore', 'Karnataka', 'India', '2023-11-11', True),
 (1, 'Customer_1', 'Hyderabad', 'Delhi', 'India', '2023-08-26', True),
 (2, 'Customer_2', 'Ahmedabad', 'West Bengal', 'India', '2023-06-23', True)]

In [64]:
# distinct() -- Transformation

In [65]:
cities_rdd = parsed_rdd.map(lambda row:row[2]).distinct()

In [67]:
cities_rdd.collect()

['Hyderabad', 'Delhi', 'Bangalore', 'Ahmedabad']

In [69]:
# take() it's returning the n number which you specify

cities_rdd.take(3)

['Hyderabad', 'Delhi', 'Bangalore']

In [70]:
#Reduce by Key - Transformation
# combines values for each key using an associative reduce function 

In [78]:
#we can chain the functions together
customers_per_city = parsed_rdd.map(lambda rows:(rows[2],1)).reduceByKey(lambda x,y:x+y)

In [79]:
customers_per_city.collect()

[('Hyderabad', 1), ('Delhi', 1), ('Bangalore', 3), ('Ahmedabad', 1)]

In [80]:
# CountByValue it an act ion

In [81]:
cust_per_city = parsed_rdd.map(lambda row :row[2]).countByValue()

In [85]:
cust_per_city

defaultdict(int, {'Bangalore': 3, 'Hyderabad': 1, 'Ahmedabad': 1, 'Delhi': 1})

# Combine more operaitons 

In [86]:
# Cities with active customer By State

In [87]:
active_cities = parsed_rdd.filter(lambda row:row[6])\
                          .map(lambda row:row[2])\
                          .distinct()
active_cities.collect()

['Hyderabad', 'Bangalore', 'Ahmedabad']

# Count Active Customer By State

In [ ]:
#run this code on a file 500mb as well

In [88]:
active_cities.saveAsTextFile("/tmp/active_cities.csv")

In [90]:
!hadoop fs -ls /tmp

Found 4 items
drwxr-xr-x   - root hadoop          0 2026-08-21 15:10 /tmp/active_cities.csv
drwxrwxrwt   - hdfs hadoop          0 2026-07-24 13:47 /tmp/hadoop-yarn
drwx-wx-wx   - hive hadoop          0 2026-07-24 13:48 /tmp/hive
-rw-r--r--   2 root hadoop         83 2026-08-18 11:38 /tmp/inputhdfsdbz.txt


In [91]:
spark.stop()